# 07 — Diagnosticity of behavioral distance for pruning-mask structure

**Research question (v2).**  Are pairwise *behavioral* distance matrices between
domain-calibrated pruned models diagnostic of *parameter-level* structure — the
overlap of their pruning masks — beyond what the pruning **level** (sparsity) and
the **pruner** already explain?

Each **variant** is one pruned model, keyed by `(pruner, domain, seed, level)`, plus
one shared **baseline** (level 0, unpruned).  Grid (per the v2 spec):

- pruners: `wanda`, `sparsegpt`
- domains (calibration sources): `math` (GSM8K), `mathqa`, `coding` (HumanEval+),
  `mbpp`, `mcq` (ARC-Challenge)
- seeds: 0, 1, 2   |   levels: 10..80 step 10   |   baseline: level 0 (one job)
- eval benchmarks: the same 5 specs, teacher-forced (top-k logprobs).

**What this notebook does.**  For every eval benchmark × distributional metric it
builds a behavioral distance matrix over all *completed* variants, builds a
mask-Jaccard distance matrix from the uploaded mask **digests**, then runs
`pruning_metrics.metrics.cluster_stats` refutation tests:

- **Mantel** — behavioral vs. mask-Jaccard correlation.
- **partial Mantel** — the same, controlling for `|level_i − level_j|` and (separately)
  a same-pruner indicator, to remove the trivial "same sparsity / same pruner ⇒ similar"
  confounds.
- **silhouette / ARI / label-permutation** on calibration-domain labels, overall and
  within `(pruner, level)` strata.

It then contrasts a **domain** pairing (`gsm8k`↔`mathqa`, same math content) with a
**format** pairing (`mathqa`↔`arc`, same MCQ format) in behavioral vs. mask space,
shows illustrative embeddings, and prints a plain-language **verdict**.

The notebook is **tolerant of partial runs**: anything missing is skipped with a
printed `NOTE`, so it runs headlessly end-to-end while the sweep is still in flight.

> Conventions mirror `04_metric_spaces.ipynb` / `05_tsne.ipynb`: caches live under
> `notebooks/experiment/results/`, downloads are skipped when already present, and the
> `per_token.json` schema is consumed unchanged via `pruning_metrics.metrics`.


## 1 · Bootstrap & configuration


In [ ]:
import os
import sys
from pathlib import Path

# Pin to the repo root so the notebook runs from anywhere.
REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").is_file():
    REPO_ROOT = REPO_ROOT.parent
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / ".env", override=False)
except ImportError:
    pass

print("REPO_ROOT =", REPO_ROOT)
print("AWS_PROFILE =", os.environ.get("AWS_PROFILE"))


In [ ]:
import json
import re

AWS_PROFILE    = os.environ.get("AWS_PROFILE", "rengz")
RESULTS_BUCKET = os.environ.get("RESULTS_BUCKET", "pruning-metrics-results-414266451290")
RESULTS_PREFIX = os.environ.get("PRUNE_EVAL_V2_PREFIX", "prune_eval_v2")

NOTEBOOK_DIR = Path.cwd() if (Path.cwd() / "experiment_config_v2.json").exists() \
    else (REPO_ROOT / "notebooks" / "experiment")
RESULTS_DIR  = NOTEBOOK_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
V2_CACHE_DIR = RESULTS_DIR / "v2_cache"
V2_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ---- distributional metrics (per_token.json consumed unchanged) ----------
METRIC_NAMES = ["kld", "jsd", "emd", "chamfer"]
METRIC_TITLES = {
    "kld": "KLD", "jsd": "JSD", "emd": "EMD", "chamfer": "Chamfer",
}

# ---- grid vocabulary (v2 spec C5) ----------------------------------------
PRUNERS = ["wanda", "sparsegpt"]
# Canonical domain keys and their content/format axes (used by the
# domain-vs-format contrast).  Whatever exact calibration-spec strings P4/P6
# emit are normalised onto these keys by _norm_domain below.
DOMAIN_META = {
    "math":   {"content": "math",    "format": "freeform", "label": "GSM8K (math)"},
    "mathqa": {"content": "math",    "format": "mcq",      "label": "MathQA (math-MCQ)"},
    "coding": {"content": "code",    "format": "freeform", "label": "HumanEval+ (code)"},
    "mbpp":   {"content": "code",    "format": "freeform", "label": "MBPP (code)"},
    "mcq":    {"content": "science", "format": "mcq",      "label": "ARC-Challenge (MCQ)"},
}
DOMAIN_COLORS = {
    "math": "tab:orange", "mathqa": "tab:red", "coding": "tab:green",
    "mbpp": "tab:olive", "mcq": "tab:blue", "baseline": "black",
}
PRUNER_MARKERS = {"wanda": "o", "sparsegpt": "s", "baseline": "*"}


def _norm_domain(raw: str) -> str:
    """Map an arbitrary calibration-spec/domain string onto a canonical key."""
    s = (raw or "").lower()
    if "mathqa" in s or "math_qa" in s:
        return "mathqa"
    if "mbpp" in s:
        return "mbpp"
    if "gsm8k" in s or s == "math" or ":math" in s or s.startswith("math"):
        return "math"
    if "humaneval" in s or "coding" in s or "code" in s:
        return "coding"
    if "arc" in s or "mcq" in s:
        return "mcq"
    return s or "unknown"


# Significance threshold used throughout the refutation tests.
ALPHA = 0.01
N_PERMUTATIONS = int(os.environ.get("V2_PERMUTATIONS", "4999"))

# ---- run scoping ---------------------------------------------------------
# Building every benchmark's pairwise matrices costs ~31 core-hours, ~93% of it
# the GSM8K chains alone. V2_BENCHES restricts the run to a comma-separated
# subset (substring match against the benchmark spec) so the analysis can be
# exercised against already-cached matrices; V2_SKIP_SYNC skips the S3 mirror
# when the local cache is known to be complete.
V2_BENCH_FILTER = [b.strip() for b in os.environ.get("V2_BENCHES", "").split(",") if b.strip()]
V2_SKIP_SYNC = os.environ.get("V2_SKIP_SYNC", "").lower() in {"1", "true", "yes"}

print("Bucket / prefix :", f"s3://{RESULTS_BUCKET}/{RESULTS_PREFIX}/")
print("Cache dir       :", V2_CACHE_DIR)
print("Permutations    :", N_PERMUTATIONS, " alpha =", ALPHA)


### Load the launch manifest

`experiment_config_v2.json` is written incrementally by the orchestration notebook
(`06_prune_eval_v2.ipynb`): one record per launched `(pruner, domain, seed)` job with
its `run_id` and results `uri`.  We read it defensively — the file may be a bare list,
or a dict keyed `launches` / `prune_eval_launches` — and skip malformed rows.


In [ ]:
cfg_path = NOTEBOOK_DIR / "experiment_config_v2.json"
launches = []
if cfg_path.exists():
    _cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
    if isinstance(_cfg, list):
        raw_launches = _cfg
    else:
        raw_launches = (
            _cfg.get("jobs")
            or _cfg.get("prune_eval_launches")
            or _cfg.get("launches")
            or _cfg.get("prune_eval_v2_launches")
            or []
        )
else:
    raw_launches = []
    print(f"NOTE: {cfg_path.name} not found — no runs to analyse. "
          "Run 06_prune_eval_v2.ipynb first. Downstream cells will no-op gracefully.")

def _run_id_from_uri(uri: str) -> str:
    return uri.rstrip("/").split("/")[-1] if uri else ""

for rec in raw_launches:
    if not isinstance(rec, dict):
        continue
    run_id = rec.get("run_id") or _run_id_from_uri(rec.get("uri", ""))
    uri = rec.get("uri") or (
        f"s3://{RESULTS_BUCKET}/{RESULTS_PREFIX}/{run_id}/" if run_id else ""
    )
    pruner = (rec.get("pruner") or "").lower()
    domain = _norm_domain(rec.get("domain") or rec.get("calibration") or "")
    seed = rec.get("seed")
    if not run_id or pruner not in PRUNERS or seed is None:
        print(f"NOTE: skipping malformed launch record: {rec!r}")
        continue
    launches.append({
        "run_id": run_id, "uri": uri, "pruner": pruner,
        "domain": domain, "seed": int(seed),
        "instance_id": rec.get("instance_id"),
    })

print(f"Launch records: {len(launches)}")
for rec in sorted(launches, key=lambda r: (r["pruner"], r["domain"], r["seed"])):
    print(f"  {rec['pruner']:>9s} | {rec['domain']:>7s} | seed={rec['seed']} | {rec['run_id']}")


## 2 · Sync digests + per-token caches from S3

For every launch we mirror two kinds of object into `results/v2_cache/<run_id>/`:

- `masks/level=NN.digest.npz` — the packed mask **digest** (small; the full masks are
  not needed for Jaccard).
- `level=NN/bench=<spec>/sample=.../per_token.json` — teacher-forced logprobs.

Existing non-empty files are skipped, so re-runs are cheap.  If AWS is unreachable the
cell prints a `NOTE` and continues on whatever is already cached — the notebook stays
runnable offline / on partial data.


In [ ]:
import concurrent.futures

def _split_uri(uri: str) -> tuple[str, str]:
    body = uri[5:]
    bucket, _, key = body.partition("/")
    return bucket, key.rstrip("/")

def _sync_one(s3, rec) -> tuple[int, int]:
    bucket, prefix = _split_uri(rec["uri"])
    local_base = V2_CACHE_DIR / rec["run_id"]
    n_pt = n_dig = 0
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents") or []:
            key = obj["Key"]
            is_pt = key.endswith("per_token.json")
            is_dig = key.endswith(".digest.npz")
            if not (is_pt or is_dig):
                continue
            rel = key[len(prefix):].lstrip("/")
            dest = local_base / rel
            if dest.exists() and dest.stat().st_size > 0:
                continue
            dest.parent.mkdir(parents=True, exist_ok=True)
            s3.download_file(bucket, key, str(dest))
            if is_pt:
                n_pt += 1
            else:
                n_dig += 1
    return n_pt, n_dig

if launches and V2_SKIP_SYNC:
    print(f"V2_SKIP_SYNC set — using the {len(launches)} run(s) already in "
          f"{V2_CACHE_DIR} without contacting S3.")
elif launches:
    try:
        import boto3
        session = boto3.session.Session(profile_name=AWS_PROFILE)
        s3 = session.client("s3")
        print(f"Syncing {len(launches)} run(s) -> {V2_CACHE_DIR} ...")
        with concurrent.futures.ThreadPoolExecutor(max_workers=8) as pool:
            futs = {pool.submit(_sync_one, s3, rec): rec for rec in launches}
            for fut in concurrent.futures.as_completed(futs):
                rec = futs[fut]
                try:
                    n_pt, n_dig = fut.result()
                    print(f"  {rec['run_id']}: +{n_pt} per_token, +{n_dig} digest")
                except Exception as exc:  # noqa: BLE001
                    print(f"  NOTE {rec['run_id']}: sync error ({exc}) — using cache only")
    except Exception as exc:  # noqa: BLE001
        print(f"NOTE: S3 unreachable ({exc}). Proceeding with whatever is cached.")
else:
    print("No launches — nothing to sync.")


## 3 · Behavioral distance matrices

We scan the local cache into an index and define a **single canonical variant order**
shared by every matrix in the notebook:

- index 0 — the shared **baseline** (level 0), if any level-0 data exists;
- then all `(pruner, domain, seed, level)` variants that have *any* cached data,
  sorted deterministically.

For each `(benchmark, metric)` we build a symmetric matrix `D` where
`D[i, j] = mean over shared tasks of metric(per_token_i, per_token_j)`.  Tasks are keyed
by the `task_id` read from inside each JSON (robust to `_safe_filename` path munging);
teacher-forced sample selection depends only on `(bench, TF_SEED, NUM_TF_SAMPLES)`, so
every variant scores the **same** tasks and rows are positionally comparable.  Matrices
cache to `results/v2_pairwise_<bench>_<metric>.npy` (+ `_meta.json`).


In [ ]:
import numpy as np
from pruning_metrics.metrics import (
    compute_kld, compute_jsd, compute_emd, compute_chamfer,
)

_METRIC_FNS = {
    "kld": compute_kld, "jsd": compute_jsd,
    "emd": compute_emd, "chamfer": compute_chamfer,
}

BASELINE_KEY = "baseline"

def _variant_key(pruner, domain, seed, level):
    if level == 0:
        return BASELINE_KEY
    return f"{pruner}|{domain}|s{seed}|L{level}"

def _run_meta(run_id):
    for rec in launches:
        if rec["run_id"] == run_id:
            return rec
    return None

# ---- scan cache: per_token index + digest paths --------------------------
# tf_index[bench][variant_key][task_id] = Path
# digest_paths[variant_key] = Path (level > 0 only)
# variant_meta[variant_key] = {pruner, domain, seed, level, is_baseline}
tf_index: dict = {}
digest_paths: dict = {}
variant_meta: dict = {}

_level_re = re.compile(r"level=(\d+)")
_bench_re = re.compile(r"bench=([^/]+)")

for rec in launches:
    run_dir = V2_CACHE_DIR / rec["run_id"]
    if not run_dir.exists():
        continue
    # digests: masks/level=NN.digest.npz
    for dig in sorted(run_dir.glob("masks/level=*.digest.npz")):
        m = _level_re.search(dig.name)
        if not m:
            continue
        level = int(m.group(1))
        if level == 0:
            continue
        vk = _variant_key(rec["pruner"], rec["domain"], rec["seed"], level)
        digest_paths[vk] = dig
        variant_meta.setdefault(vk, dict(
            pruner=rec["pruner"], domain=rec["domain"], seed=rec["seed"],
            level=level, is_baseline=False, key=vk,
        ))
    # per_token: level=NN/bench=X/sample=.../per_token.json
    for pt in sorted(run_dir.glob("level=*/bench=*/**/per_token.json")):
        rel = pt.relative_to(run_dir).as_posix()
        lm = _level_re.search(rel)
        bm = _bench_re.search(rel)
        if not (lm and bm):
            continue
        level = int(lm.group(1))
        bench = bm.group(1)
        vk = _variant_key(rec["pruner"], rec["domain"], rec["seed"], level)
        try:
            tid = json.loads(pt.read_text()).get("task_id", pt.parent.name)
        except Exception:  # noqa: BLE001
            continue
        tf_index.setdefault(bench, {}).setdefault(vk, {})[tid] = pt
        if vk == BASELINE_KEY:
            variant_meta.setdefault(vk, dict(
                pruner="baseline", domain="baseline", seed=-1,
                level=0, is_baseline=True, key=vk,
            ))
        else:
            variant_meta.setdefault(vk, dict(
                pruner=rec["pruner"], domain=rec["domain"], seed=rec["seed"],
                level=level, is_baseline=False, key=vk,
            ))

# ---- canonical variant order ---------------------------------------------
def _sort_key(vk):
    m = variant_meta[vk]
    if m["is_baseline"]:
        return (0, "", "", -1, 0)
    return (1, m["pruner"], m["domain"], m["seed"], m["level"])

VARIANTS = sorted(variant_meta.keys(), key=_sort_key)
ROW_META = [variant_meta[vk] for vk in VARIANTS]
VIDX = {vk: i for i, vk in enumerate(VARIANTS)}
BENCHES = sorted(tf_index.keys())
if V2_BENCH_FILTER:
    _all = BENCHES
    BENCHES = [b for b in _all if any(f in b for f in V2_BENCH_FILTER)]
    print(f"V2_BENCHES filter {V2_BENCH_FILTER} -> {len(BENCHES)}/{len(_all)} benchmarks")

(RESULTS_DIR / "v2_variants.json").write_text(json.dumps(ROW_META, indent=2))
print(f"Variants: {len(VARIANTS)}  (baseline present: {BASELINE_KEY in VIDX})")
print(f"Benchmarks with data: {BENCHES}")
if not VARIANTS:
    print("NOTE: no variants cached — later cells will no-op.")


In [ ]:
import multiprocessing as _mp
import os as _os


def _pairwise_task_worker(args):
    """Compute all 4 metrics for every variant pair present on one task.

    Runs in a forked worker: parses each variant's per_token.json once and
    returns compact (i, j, kld, jsd, emd, chamfer) rows.
    """
    tid, paths = args
    import json as _json
    from pruning_metrics.metrics import (
        compute_chamfer, compute_emd, compute_jsd, compute_kld,
    )
    toks = {}
    for i, p in paths.items():
        try:
            t = _json.loads(open(p).read()).get("per_token", [])
        except Exception:  # noqa: BLE001
            continue
        if t:
            toks[i] = t
    present = sorted(toks)
    rows = []
    for a in range(len(present)):
        i = present[a]
        for b in range(a + 1, len(present)):
            j = present[b]
            rows.append((
                i, j,
                compute_kld(toks[i], toks[j]),
                compute_jsd(toks[i], toks[j]),
                compute_emd(toks[i], toks[j]),
                compute_chamfer(toks[i], toks[j]),
            ))
    return rows


def _pairwise_task_worker_tagged(args):
    """Wrapper returning (tid, rows) so the parent can checkpoint task ids."""
    return args[0], _pairwise_task_worker(args)


def build_behavioral_matrices(bench):
    """All four symmetric (n, n) mean-over-tasks distance matrices for a bench.

    One parallel pass over tasks computes every metric together (a single
    JSON parse per (task, variant)), then caches per metric under the same
    file names / meta-validation scheme as the serial implementation.
    """
    n = len(VARIANTS)
    caches = {
        m: (RESULTS_DIR / f"v2_pairwise_{bench}_{m}.npy",
            RESULTS_DIR / f"v2_pairwise_{bench}_{m}_counts.npy",
            RESULTS_DIR / f"v2_pairwise_{bench}_{m}_meta.json")
        for m in METRIC_NAMES
    }

    # Cache is valid only if every metric matches the current variant order.
    loaded = {}
    for m, (c_npy, c_cnt, c_meta) in caches.items():
        try:
            if c_npy.exists() and c_meta.exists() and                     json.loads(c_meta.read_text()) == VARIANTS:
                D = np.load(c_npy)
                counts = np.load(c_cnt) if c_cnt.exists() else np.zeros((n, n), int)
                if D.shape == (n, n):
                    loaded[m] = (D, counts)
        except Exception:  # noqa: BLE001
            pass
    if len(loaded) == len(METRIC_NAMES):
        return loaded

    bench_idx = tf_index.get(bench, {})
    all_tasks = sorted({t for vk in bench_idx for t in bench_idx[vk]})
    work = []
    for tid in all_tasks:
        paths = {}
        for vk, task_map in bench_idx.items():
            p = task_map.get(tid)
            if p is not None:
                paths[VIDX[vk]] = str(p)
        if len(paths) > 1:
            work.append((tid, paths))

    D4 = {m: np.zeros((n, n), dtype=np.float64) for m in METRIC_NAMES}
    C4 = {m: np.zeros((n, n), dtype=np.int64) for m in METRIC_NAMES}

    # Restart-resilient checkpointing: harness restarts can kill this kernel
    # mid-bench, so partial accumulators are flushed every CKPT_EVERY tasks
    # and reloaded (validated against VARIANTS) on the next run.
    safe_bench = bench.replace("/", "_")
    ckpt_path = RESULTS_DIR / f"v2_ckpt_{safe_bench}.npz"
    done_tasks: set = set()
    if ckpt_path.exists():
        try:
            ck = np.load(ckpt_path, allow_pickle=True)
            if json.loads(str(ck["variants_json"])) == VARIANTS:
                for m in METRIC_NAMES:
                    D4[m] = ck[f"D_{m}"]
                    C4[m] = ck[f"C_{m}"]
                done_tasks = set(ck["done_tasks"].tolist())
                print(f"  {bench}: resuming from checkpoint "
                      f"({len(done_tasks)}/{len(work)} tasks done)")
        except Exception:  # noqa: BLE001
            done_tasks = set()

    work = [(tid, paths) for tid, paths in work if tid not in done_tasks]
    CKPT_EVERY = 25

    def _flush_ckpt():
        tmp = ckpt_path.with_suffix(".tmp.npz")
        payload = {f"D_{m}": D4[m] for m in METRIC_NAMES}
        payload.update({f"C_{m}": C4[m] for m in METRIC_NAMES})
        payload["done_tasks"] = np.array(sorted(done_tasks), dtype=object)
        payload["variants_json"] = np.array(json.dumps(VARIANTS))
        np.savez_compressed(tmp, **payload)
        tmp.replace(ckpt_path)

    workers = max(2, (_os.cpu_count() or 4) - 2)
    ctx = _mp.get_context("fork")
    tids_in_flight = {id(w): w[0] for w in work}
    completed_since_flush = 0
    with ctx.Pool(processes=workers) as pool:
        for tid, rows in pool.imap_unordered(_pairwise_task_worker_tagged, work, chunksize=1):
            for i, j, kld, jsd, emd, chamfer in rows:
                lo, hi = (i, j) if i < j else (j, i)
                for m, v in zip(METRIC_NAMES, (kld, jsd, emd, chamfer)):
                    if np.isfinite(v):
                        D4[m][lo, hi] += v
                        D4[m][hi, lo] += v
                        C4[m][lo, hi] += 1
                        C4[m][hi, lo] += 1
            done_tasks.add(tid)
            completed_since_flush += 1
            if completed_since_flush >= CKPT_EVERY:
                _flush_ckpt()
                completed_since_flush = 0
    if ckpt_path.exists():
        ckpt_path.unlink()

    out = {}
    for m in METRIC_NAMES:
        counts = C4[m]
        with np.errstate(invalid="ignore", divide="ignore"):
            D = np.where(counts > 0, D4[m] / np.where(counts > 0, counts, 1), 0.0)
        c_npy, c_cnt, c_meta = caches[m]
        np.save(c_npy, D)
        np.save(c_cnt, counts)
        c_meta.write_text(json.dumps(VARIANTS, indent=2))
        out[m] = (D, counts)
    return out


def _bench_cost(b):
    """Cheap benches first: MCQ (1-token answers) << code << GSM8K chains."""
    s = b.lower()
    if "arc" in s or "math_qa" in s or "mathqa" in s:
        return 0
    if "humaneval" in s or "mbpp" in s:
        return 1
    return 2


behavioral: dict = {}
for bench in sorted(BENCHES, key=_bench_cost):
    behavioral[bench] = {}
    per_metric = build_behavioral_matrices(bench)
    for metric_name in METRIC_NAMES:
        D, counts = per_metric[metric_name]
        behavioral[bench][metric_name] = D
        n_pairs = int((counts > 0).sum() // 2)
        print(f"  {bench}/{metric_name}: shape={D.shape}, populated pairs={n_pairs}")
if not BENCHES:
    print("NOTE: no benchmarks cached — skipping behavioral matrices.")


## 4 · Mask-Jaccard distance matrix

The mask **digests** (deterministic pseudorandom subsamples of every layer's flat mask,
positionally comparable across variants) give a parameter-space distance via
`pruning_metrics.metrics.masks.jaccard_distance` — `1 − |A∩B| / |A∪B|` over retained
positions.  The baseline (unpruned, no uploaded mask) has no digest, so it is absent from
mask space; an `avail` vector records which canonical rows carry a digest.  Cached to
`results/v2_jaccard.npy`.


In [ ]:
try:
    from pruning_metrics.metrics.masks import jaccard_matrix_packed, load_digest_packed
    _MASKS_OK = True
except Exception as exc:  # noqa: BLE001
    _MASKS_OK = False
    print(f"NOTE: masks API unavailable ({exc}); mask-space analyses will be skipped.")

n = len(VARIANTS)
jaccard_D = np.zeros((n, n), dtype=np.float64)
jaccard_avail = np.zeros(n, dtype=bool)

if _MASKS_OK and digest_paths:
    cache_npy = RESULTS_DIR / "v2_jaccard.npy"
    cache_meta = RESULTS_DIR / "v2_jaccard_meta.json"
    reuse = False
    if cache_npy.exists() and cache_meta.exists():
        try:
            meta = json.loads(cache_meta.read_text())
            if meta.get("variants") == VARIANTS:
                jaccard_D = np.load(cache_npy)
                jaccard_avail = np.array(meta["avail"], dtype=bool)
                reuse = jaccard_D.shape == (n, n)
        except Exception:  # noqa: BLE001
            reuse = False
    if not reuse:
        # Availability check only opens the zip directory (np.load on an .npz is
        # lazy), so this costs no decompression and no memory.
        def _digest_readable(path) -> bool:
            try:
                with np.load(path) as z:
                    return any(k.endswith("__bits") for k in z.files)
            except Exception:  # noqa: BLE001
                return False

        keys, paths = [], []
        for vk in VARIANTS:
            path = digest_paths.get(vk)
            if path is None:
                continue
            if _digest_readable(path):
                keys.append(vk)
                paths.append(path)
                jaccard_avail[VIDX[vk]] = True
            else:
                print(f"  NOTE: could not read digest for {vk}")

        print(f"  computing {len(keys) * (len(keys) - 1) // 2} mask-Jaccard pairs "
              f"over {len(keys)} digests (bit-packed, tiled) ...")
        # jaccard_matrix_packed loads in tiles: peak memory is set by `tile`,
        # not by the number of variants. Unpacking all of these to bool arrays
        # would need ~47 GB.
        sub = jaccard_matrix_packed(paths, tile=48)
        rows = np.array([VIDX[vk] for vk in keys])
        jaccard_D[np.ix_(rows, rows)] = sub

        np.save(cache_npy, jaccard_D)
        cache_meta.write_text(json.dumps(
            {"variants": VARIANTS, "avail": jaccard_avail.tolist()}, indent=2))
    print(f"Mask-Jaccard: {int(jaccard_avail.sum())} variants with digests.")
else:
    print("NOTE: no digests cached (or masks API unavailable) — mask space empty.")

## 5 · Refutation tests

Using `pruning_metrics.metrics.cluster_stats`, per `(benchmark, metric)`:

1. **Mantel** — correlation of behavioral and mask-Jaccard upper triangles, restricted to
   the variants present in **both** spaces.
2. **partial Mantel** — the same correlation after regressing out a **control** distance:
   (a) `|level_i − level_j|` and (b) a same-pruner indicator (0 if same pruner, else 1).
   A behavioral↔mask link that survives means behavioral distance tracks mask overlap
   *beyond* the trivial sparsity / pruner confounds.
3. **silhouette / ARI / label-permutation** on calibration-**domain** labels in behavioral
   space — overall (all variants) and within `(pruner, level)` strata — asking whether
   models separate by *what they were calibrated on*.


In [ ]:
try:
    from pruning_metrics.metrics.cluster_stats import (
        mantel, partial_mantel, silhouette_by_label,
        ari_vs_labels, label_permutation_pvalue,
    )
    _STATS_OK = True
except Exception as exc:  # noqa: BLE001
    _STATS_OK = False
    print(f"NOTE: cluster_stats API unavailable ({exc}); refutation tests skipped.")


def _submatrix(D, idx):
    idx = np.asarray(idx, dtype=int)
    return D[np.ix_(idx, idx)]

def _level_gap_matrix(idx):
    levels = np.array([ROW_META[i]["level"] for i in idx], dtype=float)
    return np.abs(levels[:, None] - levels[None, :])

def _pruner_mismatch_matrix(idx):
    pr = [ROW_META[i]["pruner"] for i in idx]
    m = np.zeros((len(idx), len(idx)))
    for a in range(len(idx)):
        for b in range(len(idx)):
            m[a, b] = 0.0 if pr[a] == pr[b] else 1.0
    return m

def _behavioral_avail(D):
    """Rows with at least one populated off-diagonal entry."""
    off = D.copy()
    np.fill_diagonal(off, 0.0)
    return np.abs(off).sum(axis=1) > 0


In [ ]:
refutation_rows = []

if _STATS_OK and BENCHES:
    for bench in BENCHES:
        for metric_name in METRIC_NAMES:
            D_beh = behavioral[bench][metric_name]
            beh_ok = _behavioral_avail(D_beh)
            # Variants present in BOTH behavioral and mask space.
            common = np.where(beh_ok & jaccard_avail)[0]
            row = {
                "bench": bench, "metric": metric_name, "n_common": int(common.size),
                "mantel_r": float("nan"), "mantel_p": float("nan"),
                "pmantel_level_r": float("nan"), "pmantel_level_p": float("nan"),
                "pmantel_pruner_r": float("nan"), "pmantel_pruner_p": float("nan"),
            }
            if common.size >= 4:
                Db = _submatrix(D_beh, common)
                Dj = _submatrix(jaccard_D, common)
                try:
                    r, p = mantel(Db, Dj, permutations=N_PERMUTATIONS, seed=0)
                    row["mantel_r"], row["mantel_p"] = float(r), float(p)
                except Exception as exc:  # noqa: BLE001
                    print(f"  NOTE mantel {bench}/{metric_name}: {exc}")
                for ctrl_name, ctrl in (
                    ("level", _level_gap_matrix(common)),
                    ("pruner", _pruner_mismatch_matrix(common)),
                ):
                    try:
                        r, p = partial_mantel(
                            Db, Dj, ctrl, permutations=N_PERMUTATIONS, seed=0)
                        row[f"pmantel_{ctrl_name}_r"] = float(r)
                        row[f"pmantel_{ctrl_name}_p"] = float(p)
                    except Exception as exc:  # noqa: BLE001
                        print(f"  NOTE partial_mantel[{ctrl_name}] "
                              f"{bench}/{metric_name}: {exc}")
            else:
                print(f"  NOTE {bench}/{metric_name}: only {common.size} variant(s) "
                      "in both spaces — need >=4 for Mantel.")
            refutation_rows.append(row)

for r in refutation_rows:
    print(f"  {r['bench']:>16s}/{r['metric']:<7s} "
          f"n={r['n_common']:>2d} "
          f"mantel r={r['mantel_r']:+.3f} p={r['mantel_p']:.4f} | "
          f"partial|level r={r['pmantel_level_r']:+.3f} p={r['pmantel_level_p']:.4f} | "
          f"partial|pruner p={r['pmantel_pruner_p']:.4f}")
if not refutation_rows:
    print("NOTE: no Mantel results (missing data or stats/masks API).")


In [ ]:
# ---- domain-label separation: overall + within (pruner, level) strata -----
def _domain_labels(idx):
    return np.array([ROW_META[i]["domain"] for i in idx])

def _domain_analysis(idx, D):
    """silhouette / ARI / permutation-p for domain labels on a behavioral submatrix."""
    labels = _domain_labels(idx)
    uniq, counts = np.unique(labels, return_counts=True)
    out = {"n": int(idx.size), "n_domains": int(uniq.size),
           "silhouette": float("nan"), "ari": float("nan"),
           "perm_p": float("nan")}
    # silhouette needs 2 <= n_labels <= n_samples - 1.
    if uniq.size < 2 or idx.size < uniq.size + 1:
        return out
    Dsub = _submatrix(D, idx)
    try:
        out["silhouette"] = float(silhouette_by_label(Dsub, labels))
    except Exception as exc:  # noqa: BLE001
        print(f"    NOTE silhouette: {exc}")
    try:
        out["ari"] = float(ari_vs_labels(Dsub, labels))
    except Exception as exc:  # noqa: BLE001
        print(f"    NOTE ari: {exc}")
    try:
        _stat, out["perm_p"] = label_permutation_pvalue(
            Dsub, labels, stat="silhouette",
            permutations=N_PERMUTATIONS, seed=0)
    except Exception as exc:  # noqa: BLE001
        print(f"    NOTE perm: {exc}")
    return out


domain_rows = []
nonbase = np.array([i for i, m in enumerate(ROW_META) if not m["is_baseline"]], dtype=int)

if _STATS_OK and BENCHES and nonbase.size:
    for bench in BENCHES:
        for metric_name in METRIC_NAMES:
            D = behavioral[bench][metric_name]
            ok = _behavioral_avail(D)
            # Overall
            idx = np.array([i for i in nonbase if ok[i]], dtype=int)
            res = _domain_analysis(idx, D)
            res.update(bench=bench, metric=metric_name, stratum="overall")
            domain_rows.append(res)
            # Within (pruner, level) strata
            strata = {}
            for i in idx:
                strata.setdefault((ROW_META[i]["pruner"], ROW_META[i]["level"]), []).append(i)
            for (pruner, level), members in sorted(strata.items()):
                sub = np.array(members, dtype=int)
                res = _domain_analysis(sub, D)
                res.update(bench=bench, metric=metric_name,
                           stratum=f"{pruner}/L{level}")
                domain_rows.append(res)

_overall = [r for r in domain_rows if r["stratum"] == "overall"]
print("Domain separation (overall, behavioral space):")
for r in _overall:
    print(f"  {r['bench']:>16s}/{r['metric']:<7s} n={r['n']:>2d} "
          f"domains={r['n_domains']} sil={r['silhouette']:+.3f} "
          f"ari={r['ari']:+.3f} perm_p={r['perm_p']:.4f}")
if not domain_rows:
    print("NOTE: no domain-separation results (missing data or stats API).")


## 6 · Domain vs. format

Do models sit closer to same-**content** peers or same-**format** peers?  Contrast a
**domain** pairing `gsm8k`↔`mathqa` (both math content) with a **format** pairing
`mathqa`↔`arc` (both MCQ format), as *mean cross-group distance* in behavioral space and
in mask space.  Smaller ⇒ closer.  If behavioral distance is driven by content, the
domain pairing is closer; if by surface format, the format pairing is.


In [ ]:
def _group_idx(domain, avail):
    return np.array(
        [i for i, m in enumerate(ROW_META)
         if not m["is_baseline"] and m["domain"] == domain and avail[i]],
        dtype=int)

def _mean_cross_group(D, idx_a, idx_b):
    if idx_a.size == 0 or idx_b.size == 0:
        return float("nan")
    block = D[np.ix_(idx_a, idx_b)]
    return float(np.mean(block))

# Pairings named in the spec (generalise via DOMAIN_META content/format axes).
PAIRINGS = [
    ("domain (math content)", "math",   "mathqa"),
    ("format (MCQ surface)",  "mathqa", "mcq"),
]

dvf_rows = []
if BENCHES:
    for bench in BENCHES:
        # behavioral: use JSD as the representative metric (bounded, symmetric),
        # falling back to the first available metric.
        metric_name = "jsd" if "jsd" in behavioral[bench] else METRIC_NAMES[0]
        Dbeh = behavioral[bench][metric_name]
        beh_ok = _behavioral_avail(Dbeh)
        for label, da, db in PAIRINGS:
            ia, ib = _group_idx(da, beh_ok), _group_idx(db, beh_ok)
            dvf_rows.append({
                "bench": bench, "space": f"behavioral/{metric_name}",
                "pairing": label, "domain_a": da, "domain_b": db,
                "mean_dist": _mean_cross_group(Dbeh, ia, ib),
                "n_a": int(ia.size), "n_b": int(ib.size),
            })
# mask space (bench-independent)
if _MASKS_OK and jaccard_avail.any():
    for label, da, db in PAIRINGS:
        ia, ib = _group_idx(da, jaccard_avail), _group_idx(db, jaccard_avail)
        dvf_rows.append({
            "bench": "(mask space)", "space": "mask-jaccard",
            "pairing": label, "domain_a": da, "domain_b": db,
            "mean_dist": _mean_cross_group(jaccard_D, ia, ib),
            "n_a": int(ia.size), "n_b": int(ib.size),
        })

print(f"{'space':>22s} {'bench':>16s} {'pairing':>22s} "
      f"{'mean_dist':>10s}  n_a n_b")
for r in dvf_rows:
    md = r["mean_dist"]
    md_s = "  nan  " if md != md else f"{md:10.4f}"
    print(f"{r['space']:>22s} {r['bench']:>16s} {r['pairing']:>22s} "
          f"{md_s}  {r['n_a']:>3d} {r['n_b']:>3d}")
if not dvf_rows:
    print("NOTE: no domain-vs-format comparison (missing data).")


## 7 · Embeddings — five reducers, with quality scores

We embed **every** `(benchmark, metric)` behavioral matrix into 2-D with five
dimension-reduction families, and colour the same points three ways — by calibration
domain, by pruning level, and by pruner — to see which axis organises the space.

| Reducer | Input | Family |
|---|---|---|
| PCA | classical-MDS coords | linear (this is PCoA) |
| t-SNE | `D` directly (`metric="precomputed"`) | neighbour-graph, local |
| UMAP | `D` directly | neighbour-graph, local |
| Isomap | `D` directly | geodesic |
| LLE | classical-MDS coords | local-linear |

**These pictures are illustrations, not tests.** t-SNE and UMAP will draw crisp,
well-separated blobs out of data with no cluster structure whatsoever, so a figure alone
cannot support a claim. Every claim in this notebook rests on the Mantel and permutation
tests in §5; this section exists to show what the space *looks* like, and to quantify how
much of that look is real.

**Reading the quality numbers.** For each embedding we report:

- **trustworthiness** — are the neighbours it drew genuine? (penalises invented structure)
- **continuity** — did it keep the true neighbours together? (penalises destroyed structure)
- **stress-1** — do 2-D distances reproduce the originals, after optimal rescaling? (lower is better)
- **Shepard ρ** — does it at least get the *ordering* of distances right? (higher is better)

High trustworthiness with low Shepard ρ — the usual t-SNE/UMAP signature — means local
neighbourhoods are trustworthy but between-group distances are not. In that regime
"these two clusters are far apart" is **not** a supportable reading of the picture.

**Two caveats specific to this data.**

1. *Unobserved pairs.* A pair of variants that never shared an evaluation task is stored
   as distance `0.0`, which is indistinguishable from "identical". A single such row puts
   one point at distance 0 from everything and collapses Isomap's geodesic graph outright,
   so we restrict each matrix to its largest fully-observed submatrix first.
2. *Non-Euclidean distances.* PCA and LLE have no precomputed mode, so they run on
   classical-MDS coordinates — and that step **discards** the negative eigenvalues of the
   double-centred Gram matrix. For KLD that throws away roughly a quarter of the
   eigenvalue mass and over half the dimensions, so its PCA/LLE panels are embedding a
   materially different object than its t-SNE/UMAP/Isomap panels. We report the discarded
   fraction as `mds_neg_ratio` on every row and flag it in the figure titles rather than
   quietly patching `D` with an additive constant, which would change what is embedded.

In [ ]:
import csv
import time

from pruning_metrics.embedding import (
    REDUCERS, complete_submatrix_indices, embed_2d, mds_spectrum,
)
from pruning_metrics.metrics.embedding_quality import embedding_quality

EMB_DIR     = RESULTS_DIR / "v2_embeddings"          # cached coordinate arrays
EMB_FIG_DIR = RESULTS_DIR / "v2_embedding_figures"   # figures (driver.py globs *_figures/)
EMB_DIR.mkdir(parents=True, exist_ok=True)
EMB_FIG_DIR.mkdir(parents=True, exist_ok=True)

QUALITY_K  = 12          # ~half the 24 variants sharing a (pruner, domain) cell
QUALITY_KS = (5, 12, 25)  # reported in the CSV to show k-sensitivity
MIN_POINTS = 10           # below this an embedding is not worth computing


def _safe(name: str) -> str:
    """Filename-safe benchmark key (bench specs contain '/' and ':')."""
    return name.replace("/", "_")


# (bench, metric, reducer) -> (coords, kept_row_indices)
embeddings: dict = {}
# (bench, metric) -> mds_spectrum of the restricted matrix
mds_info: dict = {}
quality_rows: list = []

for bench in BENCHES:
    for metric_name in METRIC_NAMES:
        D_full = behavioral[bench][metric_name]
        # Drop rows whose pairs were never observed; see caveat 1 above.
        idx = complete_submatrix_indices(D_full)
        if idx.size < MIN_POINTS:
            print(f"  {bench}/{metric_name}: only {idx.size} usable variants — skipped.")
            continue
        D = _submatrix(D_full, idx)
        spec = mds_info[(bench, metric_name)] = mds_spectrum(D)
        dropped = D_full.shape[0] - idx.size
        print(
            f"  {bench}/{metric_name}: n={idx.size} (dropped {dropped}), "
            f"mds_dims={spec['n_pos']}, neg_ratio={spec['neg_ratio']:.4f}"
        )

        for reducer in REDUCERS:
            cache_npy  = EMB_DIR / f"emb_{_safe(bench)}_{metric_name}_{reducer}.npy"
            cache_meta = cache_npy.with_suffix(".json")
            coords = None
            if cache_npy.exists() and cache_meta.exists():
                try:
                    meta = json.loads(cache_meta.read_text())
                    if meta.get("variants") == VARIANTS and meta.get("idx") == idx.tolist():
                        cached = np.load(cache_npy)
                        if cached.shape == (idx.size, 2):
                            coords, info, seconds = cached, meta["info"], meta["seconds"]
                except Exception:  # noqa: BLE001
                    coords = None

            if coords is None:
                t0 = time.time()
                coords, info = embed_2d(D, reducer)
                seconds = time.time() - t0
                np.save(cache_npy, coords)
                cache_meta.write_text(json.dumps(
                    {"variants": VARIANTS, "idx": idx.tolist(),
                     "info": info, "seconds": seconds}, indent=2, default=str))

            embeddings[(bench, metric_name, reducer)] = (coords, idx)

            row = {
                "bench": bench, "metric": metric_name, "reducer": reducer,
                "mds_neg_ratio": round(spec["neg_ratio"], 6),
                "mds_dims": spec["n_pos"], "mds_var_2d": round(spec["var_2d"], 6),
                "params_json": json.dumps(info.get("params", {}), default=str),
                "seconds": round(seconds, 3),
            }
            for k in QUALITY_KS:
                q = embedding_quality(D, coords, k=k)
                if k == QUALITY_K:
                    row.update(n=q["n"], k=q["k"],
                               trust=round(q["trustworthiness"], 4),
                               cont=round(q["continuity"], 4),
                               stress1=round(q["stress1"], 4),
                               alpha=round(q["alpha"], 6),
                               shepard_rho=round(q["shepard_rho"], 4))
                else:
                    row[f"trust_k{k}"] = round(q["trustworthiness"], 4)
                    row[f"cont_k{k}"]  = round(q["continuity"], 4)
            quality_rows.append(row)

QUALITY_CSV = RESULTS_DIR / "v2_embedding_quality.csv"
if quality_rows:
    fields = ["bench", "metric", "reducer", "n", "k", "trust", "cont", "stress1",
              "alpha", "shepard_rho", "mds_neg_ratio", "mds_dims", "mds_var_2d"]
    fields += [f for k in QUALITY_KS if k != QUALITY_K
               for f in (f"trust_k{k}", f"cont_k{k}")]
    fields += ["params_json", "seconds"]
    with QUALITY_CSV.open("w", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=fields)
        writer.writeheader()
        writer.writerows(quality_rows)
    print(f"\n{len(embeddings)} embeddings -> {EMB_DIR}")
    print(f"quality table ({len(quality_rows)} rows) -> {QUALITY_CSV}")
else:
    print("NOTE: no embeddings computed — no behavioral matrices available.")

In [ ]:
# ---- quality table, printed inline -------------------------------------------
if quality_rows:
    hdr = (f"{'bench':<38s} {'metric':<8s} {'reducer':<7s} {'n':>4s} "
           f"{'trust':>6s} {'cont':>6s} {'stress':>6s} {'shepard':>7s} {'negR':>6s}")
    print(hdr)
    print("-" * len(hdr))
    for r in quality_rows:
        flag = " *" if r["mds_neg_ratio"] > 0.05 and r["reducer"] in ("pca", "lle") else ""
        print(f"{r['bench'][:38]:<38s} {r['metric']:<8s} {r['reducer']:<7s} {r['n']:>4d} "
              f"{r['trust']:>6.3f} {r['cont']:>6.3f} {r['stress1']:>6.3f} "
              f"{r['shepard_rho']:>7.3f} {r['mds_neg_ratio']:>6.3f}{flag}")
    print("\n* = PCA/LLE on a strongly non-Euclidean matrix; classical MDS discarded "
          ">5% of the eigenvalue mass, so treat that panel with extra caution.")

### Figures

Five reducers × 5 benchmarks × 4 metrics is 100 embeddings, which is far too many
pictures to read one at a time. Instead we emit:

1. **`embedding_grid_{bench}_{metric}.png`** — 5 reducers (rows) × 3 colourings (columns).
   The "do the five reducers agree?" view, produced for every `(bench, metric)`.
2. **`summary_{reducer}_by_{domain,level}.png`** — benchmarks (rows) × metrics (columns),
   one sheet per reducer per colouring. Ten sheets covering all 100 embeddings.
3. **`embedding_quality_heatmap.png`** — the four quality measures at a glance.
4. **`sensitivity_{bench}_{metric}.png`** — the same data under a sweep of t-SNE
   perplexity and UMAP `n_neighbors`, which is the honest way to show how much of an
   apparent cluster is a hyperparameter artefact.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

PRUNER_COLORS = {"wanda": "tab:purple", "sparsegpt": "tab:cyan", "baseline": "black"}
_BASE_STYLE = dict(marker="*", s=340, c="black", edgecolors="white",
                   linewidths=0.7, zorder=5)
_DOT_STYLE  = dict(s=45, alpha=0.85, edgecolors="k", linewidths=0.3, zorder=3)


def _meta_for(idx):
    """Row metadata for the retained rows of an embedding."""
    return [ROW_META[i] for i in idx]


def _split_baseline(meta):
    base = [i for i, m in enumerate(meta) if m["is_baseline"]]
    rest = [i for i, m in enumerate(meta) if not m["is_baseline"]]
    return base, rest


def _draw_by_domain(ax, Y, meta):
    base, rest = _split_baseline(meta)
    for dom in sorted({meta[i]["domain"] for i in rest}):
        sel = [i for i in rest if meta[i]["domain"] == dom]
        # One vectorised call per group: a per-point loop would run 232x per panel.
        ax.scatter(Y[sel, 0], Y[sel, 1], c=DOMAIN_COLORS.get(dom, "gray"), **_DOT_STYLE)
    if base:
        ax.scatter(Y[base, 0], Y[base, 1], **_BASE_STYLE)


def _draw_by_level(ax, Y, meta):
    base, rest = _split_baseline(meta)
    levels = np.array([meta[i]["level"] for i in rest], dtype=float)
    sc = ax.scatter(Y[rest, 0], Y[rest, 1], c=levels, cmap="viridis", **_DOT_STYLE)
    if base:
        ax.scatter(Y[base, 0], Y[base, 1], **_BASE_STYLE)
    return sc


def _draw_by_pruner(ax, Y, meta):
    base, rest = _split_baseline(meta)
    for pruner in sorted({meta[i]["pruner"] for i in rest}):
        sel = [i for i in rest if meta[i]["pruner"] == pruner]
        ax.scatter(Y[sel, 0], Y[sel, 1], c=PRUNER_COLORS.get(pruner, "gray"),
                   marker=PRUNER_MARKERS.get(pruner, "o"), **_DOT_STYLE)
    if base:
        ax.scatter(Y[base, 0], Y[base, 1], **_BASE_STYLE)


_DRAW = {"domain": _draw_by_domain, "level": _draw_by_level, "pruner": _draw_by_pruner}


def _blank_ticks(ax):
    ax.set_xticks([]); ax.set_yticks([])


def _domain_handles(meta):
    doms = sorted({m["domain"] for m in meta if not m["is_baseline"]})
    return [Line2D([], [], marker="o", linestyle="None", markeredgecolor="k",
                   markerfacecolor=DOMAIN_COLORS.get(d, "gray"),
                   label=DOMAIN_META.get(d, {}).get("label", d)) for d in doms] + \
           [Line2D([], [], marker="*", linestyle="None", markersize=13,
                   markerfacecolor="black", markeredgecolor="white", label="baseline")]


def _pruner_handles(meta):
    prs = sorted({m["pruner"] for m in meta if not m["is_baseline"]})
    return [Line2D([], [], marker=PRUNER_MARKERS.get(p, "o"), linestyle="None",
                   markeredgecolor="k", markerfacecolor=PRUNER_COLORS.get(p, "gray"),
                   label=p) for p in prs]


def _quality_of(bench, metric_name, reducer):
    for r in quality_rows:
        if (r["bench"], r["metric"], r["reducer"]) == (bench, metric_name, reducer):
            return r
    return None


def _panel_title(bench, metric_name, reducer):
    q = _quality_of(bench, metric_name, reducer)
    spec = REDUCERS[reducer]
    if q is None:
        return spec.title
    warn = " ⚠" if (q["mds_neg_ratio"] > 0.05 and spec.needs_coords) else ""
    return f"{spec.title}{warn}\nT={q['trust']:.2f}  ρ={q['shepard_rho']:.2f}"


print("Figure helpers defined.")

In [ ]:
# ---- 1. per-(bench, metric) grid: reducers x colourings ----------------------
# Headline = the combo whose distances are closest to Euclidean. There all five
# reducers are embedding the same object, so any disagreement between them is a
# fact about the algorithms rather than about what classical MDS threw away.
HEADLINE = min(mds_info, key=lambda c: mds_info[c]["neg_ratio"]) if mds_info else None

for bench in BENCHES:
    for metric_name in METRIC_NAMES:
        present = [r for r in REDUCERS if (bench, metric_name, r) in embeddings]
        if not present:
            continue
        _, idx = embeddings[(bench, metric_name, present[0])]
        meta = _meta_for(idx)

        fig, axes = plt.subplots(len(present), 3,
                                 figsize=(15, 4.6 * len(present)), squeeze=False)
        neg = mds_info[(bench, metric_name)]["neg_ratio"]
        fig.suptitle(
            f"{bench} — {METRIC_TITLES.get(metric_name, metric_name)}   "
            f"(n={idx.size} variants, MDS neg/pos={neg:.3f})\n"
            f"illustration only — claims rest on the tests in §5",
            fontsize=13, fontweight="bold",
        )
        for row, reducer in enumerate(present):
            Y, _ = embeddings[(bench, metric_name, reducer)]
            for col, colouring in enumerate(("domain", "level", "pruner")):
                ax = axes[row, col]
                out = _DRAW[colouring](ax, Y, meta)
                _blank_ticks(ax)
                if col == 0:
                    ax.set_ylabel(_panel_title(bench, metric_name, reducer),
                                  fontsize=9, fontweight="bold")
                if row == 0:
                    ax.set_title(f"by {colouring}", fontsize=11, fontweight="bold")
                if colouring == "level" and row == 0:
                    fig.colorbar(out, ax=ax, fraction=0.046, label="level (%)")

        fig.legend(handles=_domain_handles(meta) + _pruner_handles(meta),
                   loc="lower center", ncol=8, fontsize=8, frameon=False,
                   bbox_to_anchor=(0.5, -0.01))
        fig.tight_layout(rect=(0, 0.02, 1, 0.97))
        out_path = EMB_FIG_DIR / f"embedding_grid_{_safe(bench)}_{metric_name}.png"
        fig.savefig(out_path, dpi=130, bbox_inches="tight")
        if (bench, metric_name) == HEADLINE:
            plt.show()
        else:
            plt.close(fig)   # keep the executed notebook small
        print(f"  saved {out_path.name}")

In [ ]:
# ---- 2. per-reducer summary sheets: benchmarks x metrics ---------------------
for reducer in REDUCERS:
    for colouring in ("domain", "level"):
        combos = [(b, m) for b in BENCHES for m in METRIC_NAMES
                  if (b, m, reducer) in embeddings]
        if not combos:
            continue
        rows = sorted({b for b, _ in combos})
        fig, axes = plt.subplots(len(rows), len(METRIC_NAMES),
                                 figsize=(4.4 * len(METRIC_NAMES), 4.4 * len(rows)),
                                 squeeze=False)
        fig.suptitle(f"{REDUCERS[reducer].title} — coloured by {colouring}",
                     fontsize=14, fontweight="bold")
        meta = None
        for r, bench in enumerate(rows):
            for c, metric_name in enumerate(METRIC_NAMES):
                ax = axes[r, c]
                _blank_ticks(ax)
                if (bench, metric_name, reducer) not in embeddings:
                    ax.set_visible(False)
                    continue
                Y, idx = embeddings[(bench, metric_name, reducer)]
                meta = _meta_for(idx)
                _DRAW[colouring](ax, Y, meta)
                q = _quality_of(bench, metric_name, reducer)
                ax.set_title(f"{METRIC_TITLES.get(metric_name, metric_name)}"
                             + (f"   T={q['trust']:.2f} ρ={q['shepard_rho']:.2f}"
                                if q else ""), fontsize=9)
                if c == 0:
                    ax.set_ylabel(bench.split(":")[0], fontsize=10, fontweight="bold")
        if meta is not None and colouring == "domain":
            fig.legend(handles=_domain_handles(meta), loc="lower center",
                       ncol=7, fontsize=8, frameon=False, bbox_to_anchor=(0.5, -0.02))
        fig.tight_layout(rect=(0, 0.02, 1, 0.96))
        out_path = EMB_FIG_DIR / f"summary_{reducer}_by_{colouring}.png"
        fig.savefig(out_path, dpi=120, bbox_inches="tight")
        plt.close(fig)
        print(f"  saved {out_path.name}")

In [ ]:
# ---- 3. quality heatmap: the figure that answers "is any of this real?" ------
if quality_rows:
    combos  = sorted({(r["bench"], r["metric"]) for r in quality_rows})
    reducers = [r for r in REDUCERS if any(q["reducer"] == r for q in quality_rows)]
    panels = [
        ("trust",       "Trustworthiness (higher = fewer invented neighbours)", "viridis", 0, 1),
        ("cont",        "Continuity (higher = fewer torn-apart neighbours)",    "viridis", 0, 1),
        ("stress1",     "Kruskal stress-1 (lower = truer distances)",           "magma_r", 0, 1),
        ("shepard_rho", "Shepard ρ (higher = truer distance ordering)",         "viridis", -1, 1),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(6 + 1.6 * len(reducers),
                                           2.2 + 0.42 * len(combos) * 2))
    lookup = {(r["bench"], r["metric"], r["reducer"]): r for r in quality_rows}
    for ax, (field, title, cmap, lo, hi) in zip(axes.ravel(), panels):
        M = np.full((len(combos), len(reducers)), np.nan)
        for i, (b, m) in enumerate(combos):
            for j, red in enumerate(reducers):
                row = lookup.get((b, m, red))
                if row is not None:
                    M[i, j] = row[field]
        im = ax.imshow(M, cmap=cmap, vmin=lo, vmax=hi, aspect="auto")
        ax.set_xticks(range(len(reducers)))
        ax.set_xticklabels(reducers, fontsize=9)
        ax.set_yticks(range(len(combos)))
        ax.set_yticklabels([f"{b.split(':')[0]}/{m}" for b, m in combos], fontsize=8)
        ax.set_title(title, fontsize=10, fontweight="bold")
        for i in range(len(combos)):
            for j in range(len(reducers)):
                if np.isfinite(M[i, j]):
                    ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center",
                            fontsize=7.5, color="w" if cmap == "magma_r" else "k")
        fig.colorbar(im, ax=ax, fraction=0.046)
    fig.suptitle("Embedding quality — how much of each picture is real",
                 fontsize=13, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    out_path = EMB_FIG_DIR / "embedding_quality_heatmap.png"
    fig.savefig(out_path, dpi=140, bbox_inches="tight")
    plt.show()
    print(f"  saved {out_path.name}")

In [ ]:
# ---- 4. hyperparameter sensitivity ------------------------------------------
# t-SNE and UMAP have no "correct" neighbourhood size, and the apparent cluster
# structure moves with it. Showing the sweep is more honest than picking one.
TSNE_PERPLEXITIES = [5, 10, 30, 50, 100]
UMAP_NEIGHBOURS   = [5, 15, 50, 100]

if HEADLINE is not None:
    bench, metric_name = HEADLINE
    _, idx = embeddings[(bench, metric_name, "tsne")]
    D = _submatrix(behavioral[bench][metric_name], idx)
    meta = _meta_for(idx)

    ncol = max(len(TSNE_PERPLEXITIES), len(UMAP_NEIGHBOURS))
    fig, axes = plt.subplots(2, ncol, figsize=(3.5 * ncol, 7.6), squeeze=False)
    fig.suptitle(
        f"Hyperparameter sensitivity — {bench} / "
        f"{METRIC_TITLES.get(metric_name, metric_name)}, coloured by domain\n"
        "if the grouping changes with the setting, it is an artefact, not a finding",
        fontsize=12, fontweight="bold",
    )
    for col in range(ncol):
        for row, (reducer, values, kw) in enumerate(
            (("tsne", TSNE_PERPLEXITIES, "perplexity"),
             ("umap", UMAP_NEIGHBOURS, "n_neighbors")),
        ):
            ax = axes[row, col]
            _blank_ticks(ax)
            if col >= len(values):
                ax.set_visible(False)
                continue
            Y, _ = embed_2d(D, reducer, **{kw: values[col]})
            _draw_by_domain(ax, Y, meta)
            q = embedding_quality(D, Y, k=QUALITY_K)
            ax.set_title(f"{REDUCERS[reducer].title}  {kw}={values[col]}\n"
                         f"T={q['trustworthiness']:.2f}  ρ={q['shepard_rho']:.2f}",
                         fontsize=9)
    fig.legend(handles=_domain_handles(meta), loc="lower center", ncol=7,
               fontsize=8, frameon=False, bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout(rect=(0, 0.02, 1, 0.93))
    out_path = EMB_FIG_DIR / f"sensitivity_{_safe(bench)}_{metric_name}.png"
    fig.savefig(out_path, dpi=130, bbox_inches="tight")
    plt.show()
    print(f"  saved {out_path.name}")

## 8 · Does the picture predict damage? (R²)

The quality scores in §7 ask whether an embedding is faithful to the *distance
matrix*. This section asks something stronger and more useful: whether the
embedding is faithful to **reality**.

For each network we take how far the reducer moved it from the unpruned
baseline, and regress the network's **actual degradation** on it. If a picture
is worth reading, networks drawn further from the baseline should be the ones
that actually got worse.

**Degradation for v2 is log-perplexity increase**, `log ppl(variant) − log
ppl(baseline)`, taken from each run's `summary.json`. The v2 sweep is
teacher-forced only — it never generated free-form answers — so there is no
pass@1 here. (Notebook 05 runs the same regression against real accuracy drop
on the 13 v1 models.) Since `perplexity = exp(−mean_logprob)` exactly in these
records, this is just a difference of mean log-probabilities, which is the
natural linear scale. Level-0 `mean_logprob` is identical across all runs, so
the baseline is genuinely shared.

Each panel also carries a **raw** control: the same regression using distance
from the baseline in the original 232×232 matrix instead of in the 2-D picture.
The gap between a reducer's R² and the raw R² is what the reduction to two
dimensions cost in predictive power.

In [ ]:
from pruning_metrics.metrics.embedding_quality import baseline_distances, linear_r2

# ---- per-variant degradation from the run summaries ------------------------
# summary.json spells benchmarks with "/" (math:openai/gsm8k:main) while the
# cache paths -- and therefore BENCHES -- use "_" (math:openai_gsm8k:main).
def _summary_bench_key(raw: str) -> str:
    return raw.replace("/", "_")

degradation: dict = {}   # variant_key -> {bench: log-perplexity increase}
baseline_logprob: dict = {}   # bench -> level-0 mean_logprob
n_summaries = 0

for rec in launches:
    path = V2_CACHE_DIR / rec["run_id"] / "summary.json"
    if not path.exists():
        continue
    try:
        levels = json.loads(path.read_text()).get("levels", {})
    except Exception:  # noqa: BLE001
        continue
    n_summaries += 1
    for raw_bench, entry in (levels.get("0") or {}).items():
        baseline_logprob.setdefault(_summary_bench_key(raw_bench), entry["mean_logprob"])
    for level_str, per_bench in levels.items():
        level = int(level_str)
        if level == 0:
            continue
        vk = _variant_key(rec["pruner"], rec["domain"], rec["seed"], level)
        for raw_bench, entry in per_bench.items():
            bench = _summary_bench_key(raw_bench)
            base = baseline_logprob.get(bench)
            if base is None:
                continue
            # log ppl(v) - log ppl(0) == mean_logprob(0) - mean_logprob(v)
            degradation.setdefault(vk, {})[bench] = base - entry["mean_logprob"]

degradation[BASELINE_KEY] = {b: 0.0 for b in baseline_logprob}
covered = sum(1 for vk in VARIANTS if vk in degradation)
print(f"Read {n_summaries} run summaries; degradation known for "
      f"{covered}/{len(VARIANTS)} variants across {len(baseline_logprob)} benchmarks.")

In [ ]:
# ---- regress degradation on embedding radius, per (bench, metric, reducer) --
r2_rows: list = []

for bench in BENCHES:
    for metric_name in METRIC_NAMES:
        key0 = (bench, metric_name, next(iter(REDUCERS)))
        if key0 not in embeddings:
            continue
        _, idx = embeddings[key0]
        local_base = [i for i, row in enumerate(idx) if ROW_META[row]["is_baseline"]]
        if not local_base:
            print(f"  {bench}/{metric_name}: baseline not in the usable submatrix — skipped.")
            continue
        b = local_base[0]

        # Degradation on THIS eval benchmark, aligned to the retained rows.
        y = np.array([
            degradation.get(VARIANTS[row], {}).get(bench, np.nan) for row in idx
        ], dtype=float)

        # Control: distance from baseline in the original distance matrix.
        D = _submatrix(behavioral[bench][metric_name], idx)
        raw = linear_r2(D[b], y)
        r2_rows.append(dict(bench=bench, metric=metric_name, reducer="raw",
                            **{k: raw[k] for k in ("n", "r2", "r", "slope")}))

        for reducer in REDUCERS:
            if (bench, metric_name, reducer) not in embeddings:
                continue
            Y, _ = embeddings[(bench, metric_name, reducer)]
            fit = linear_r2(baseline_distances(Y, b), y)
            r2_rows.append(dict(bench=bench, metric=metric_name, reducer=reducer,
                                **{k: fit[k] for k in ("n", "r2", "r", "slope")}))

R2_CSV = RESULTS_DIR / "v2_embedding_r2.csv"
if r2_rows:
    with R2_CSV.open("w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=["bench", "metric", "reducer", "n", "r2", "r", "slope"])
        w.writeheader()
        w.writerows(r2_rows)
    print(f"{len(r2_rows)} rows -> {R2_CSV}\n")

    hdr = f"{'bench':<22s} {'metric':<8s} " + " ".join(f"{r:>7s}" for r in ["raw", *REDUCERS])
    print(hdr); print("-" * len(hdr))
    for bench in BENCHES:
        for metric_name in METRIC_NAMES:
            cells = []
            for red in ["raw", *REDUCERS]:
                hit = [x for x in r2_rows
                       if (x["bench"], x["metric"], x["reducer"]) == (bench, metric_name, red)]
                cells.append(f"{hit[0]['r2']:7.3f}" if hit and hit[0]["r2"] == hit[0]["r2"] else "      -")
            print(f"{bench.split(':')[0]:<22s} {metric_name:<8s} " + " ".join(cells))
else:
    print("NOTE: no embeddings available — R² section skipped.")

In [ ]:
# ---- figure: networks in radius-vs-degradation space ------------------------
if r2_rows and HEADLINE is not None:
    bench, metric_name = HEADLINE
    _, idx = embeddings[(bench, metric_name, next(iter(REDUCERS)))]
    meta = _meta_for(idx)
    b = [i for i, row in enumerate(idx) if ROW_META[row]["is_baseline"]][0]
    y = np.array([degradation.get(VARIANTS[row], {}).get(bench, np.nan) for row in idx])
    D = _submatrix(behavioral[bench][metric_name], idx)

    panels = [("raw", "raw distance in the 232×232 matrix", D[b])]
    panels += [(r, REDUCERS[r].title, baseline_distances(embeddings[(bench, metric_name, r)][0], b))
               for r in REDUCERS if (bench, metric_name, r) in embeddings]

    fig, axes = plt.subplots(2, 3, figsize=(17, 10))
    fig.suptitle(
        f"Does distance-from-baseline predict damage?  {bench} / "
        f"{METRIC_TITLES.get(metric_name, metric_name)}\n"
        "x = distance from the unpruned baseline    y = log-perplexity increase",
        fontsize=13, fontweight="bold",
    )
    for ax, (key, title, x) in zip(axes.ravel(), panels):
        for dom in sorted({m["domain"] for m in meta if not m["is_baseline"]}):
            sel = [i for i, m in enumerate(meta) if m["domain"] == dom and not m["is_baseline"]]
            ax.scatter(x[sel], y[sel], c=DOMAIN_COLORS.get(dom, "gray"),
                       s=42, alpha=0.85, edgecolors="k", linewidths=0.3)
        ax.scatter(x[b], y[b], **_BASE_STYLE)
        fit = linear_r2(x, y)
        if fit["r2"] == fit["r2"]:
            ok = np.isfinite(x) & np.isfinite(y)
            xs = np.linspace(x[ok].min(), x[ok].max(), 50)
            ax.plot(xs, fit["slope"] * xs + fit["intercept"],
                    color="k", lw=1.2, ls="--", alpha=0.7)
        ax.set_title(f"{title}\nR²={fit['r2']:.3f}  (n={fit['n']})", fontsize=10, fontweight="bold")
        ax.set_xlabel("distance from baseline", fontsize=9)
        ax.set_ylabel("log-perplexity increase", fontsize=9)
    for ax in axes.ravel()[len(panels):]:
        ax.set_visible(False)
    fig.legend(handles=_domain_handles(meta), loc="lower center", ncol=7,
               fontsize=8, frameon=False, bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout(rect=(0, 0.03, 1, 0.93))
    out = EMB_FIG_DIR / f"r2_radius_vs_degradation_{_safe(bench)}_{metric_name}.png"
    fig.savefig(out, dpi=140, bbox_inches="tight")
    plt.show()
    print(f"  saved {out.name}")

    # Heatmap across every (bench, metric) x reducer.
    combos = sorted({(x["bench"], x["metric"]) for x in r2_rows})
    cols = ["raw", *REDUCERS]
    M = np.full((len(combos), len(cols)), np.nan)
    lookup = {(x["bench"], x["metric"], x["reducer"]): x["r2"] for x in r2_rows}
    for i, (bb, mm) in enumerate(combos):
        for j, red in enumerate(cols):
            M[i, j] = lookup.get((bb, mm, red), np.nan)
    fig, ax = plt.subplots(figsize=(2 + 1.5 * len(cols), 2 + 0.42 * len(combos)))
    im = ax.imshow(M, cmap="viridis", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, fontsize=9)
    ax.set_yticks(range(len(combos)))
    ax.set_yticklabels([f"{b.split(':')[0]}/{m}" for b, m in combos], fontsize=8)
    for i in range(len(combos)):
        for j in range(len(cols)):
            if np.isfinite(M[i, j]):
                ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=7.5)
    ax.set_title("R² of log-perplexity increase on distance from baseline\n"
                 "('raw' = full distance matrix, the ceiling each reducer is trying to keep)",
                 fontsize=11, fontweight="bold")
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    out = EMB_FIG_DIR / "r2_heatmap.png"
    fig.savefig(out, dpi=140, bbox_inches="tight")
    plt.show()
    print(f"  saved {out.name}")

## 9 · Verdict

The summary table gives, per `(benchmark, metric)`: Mantel `r`/`p`, partial-Mantel
`r`/`p` (controlling `|Δlevel|`), silhouette, ARI, and the domain permutation `p`.

**Decision rule.**  The diagnosticity claim — *behavioral distance is diagnostic of
pruning-mask structure beyond sparsity, and models separate by calibration domain* — is
judged **supported** when partial-Mantel `p < 0.01` for a **majority** of `(bench, metric)`
combos **and** the overall domain-silhouette permutation `p < 0.01`; **refuted** when
neither holds; **mixed** otherwise.


In [ ]:
# Join refutation (Mantel/partial) with the overall domain-separation results.
_dom_overall = {
    (r["bench"], r["metric"]): r
    for r in domain_rows if r["stratum"] == "overall"
}

summary = []
for r in refutation_rows:
    key = (r["bench"], r["metric"])
    dom = _dom_overall.get(key, {})
    summary.append({
        **r,
        "silhouette": dom.get("silhouette", float("nan")),
        "ari": dom.get("ari", float("nan")),
        "domain_perm_p": dom.get("perm_p", float("nan")),
    })

# Header
cols = f"{'bench':>16s} {'metric':>7s} {'n':>3s} {'mantel_r':>9s} {'mantel_p':>9s} " \
       f"{'pM|lvl_r':>9s} {'pM|lvl_p':>9s} {'silhou':>7s} {'ari':>7s} {'dom_p':>7s}"
print(cols)
print("-" * len(cols))

def _f(x, w=9, p=3):
    return (" " * (w - 3) + "nan") if x != x else f"{x:{w}.{p}f}"

for s in summary:
    print(f"{s['bench']:>16s} {s['metric']:>7s} {s['n_common']:>3d} "
          f"{_f(s['mantel_r'])} {_f(s['mantel_p'],9,4)} "
          f"{_f(s['pmantel_level_r'])} {_f(s['pmantel_level_p'],9,4)} "
          f"{_f(s['silhouette'],7)} {_f(s['ari'],7)} {_f(s['domain_perm_p'],7,4)}")

# ---- decision rule -------------------------------------------------------
# A combo only counts as supporting diagnosticity if the partial-Mantel
# correlation is BOTH significant and POSITIVE: partial_mantel returns a
# two-sided |r| p-value, and a significantly negative r (behavioral distance
# anti-tracking mask distance) is evidence against the claim, not for it.
partial_rp = [
    (s["pmantel_level_r"], s["pmantel_level_p"])
    for s in summary
    if s["pmantel_level_p"] == s["pmantel_level_p"]
]
partial_ps = [p for _r, p in partial_rp]
n_partial_sig = sum(1 for r, p in partial_rp if p < ALPHA and r > 0)
majority_partial_sig = bool(partial_rp) and n_partial_sig > len(partial_rp) / 2

dom_ps = [r["perm_p"] for r in _dom_overall.values() if r["perm_p"] == r["perm_p"]]
n_dom_sig = sum(1 for p in dom_ps if p < ALPHA)
domain_sig = bool(dom_ps) and n_dom_sig > len(dom_ps) / 2

if majority_partial_sig and domain_sig:
    verdict = "SUPPORTED"
elif not majority_partial_sig and not domain_sig:
    verdict = "REFUTED"
else:
    verdict = "MIXED"

print()
print("=" * 72)
print("VERDICT — behavioral distance as a diagnostic of pruning-mask structure")
print("=" * 72)
if not summary:
    print("INSUFFICIENT DATA: no completed (bench, metric) combos yet. Re-run once the")
    print("sweep has produced masks + per_token records for several variants.")
else:
    print(f"  partial-Mantel (control |Delta level|) positive and significant at p<{ALPHA}: "
          f"{n_partial_sig}/{len(partial_ps)} combos "
          f"-> majority = {majority_partial_sig}")
    print(f"  domain-silhouette permutation significant at p<{ALPHA}: "
          f"{n_dom_sig}/{len(dom_ps)} benches -> {domain_sig}")
    print()
    print(f"  ==> The diagnosticity claim is: {verdict}")
    print()
    if verdict == "SUPPORTED":
        print("  Behavioral distance between pruned models tracks pruning-mask overlap")
        print("  even after removing the sparsity-level confound, and models separate by")
        print("  calibration domain: behavioral geometry is diagnostic of parameter-level")
        print("  structure.")
    elif verdict == "REFUTED":
        print("  Once |Delta level| is controlled the behavioral<->mask link vanishes and")
        print("  domains do not separate: behavioral distance reflects sparsity, not the")
        print("  specific pruning mask. The diagnosticity claim is not supported.")
    else:
        print("  Evidence is mixed: one of {partial-Mantel majority, domain separation}")
        print("  holds but not both. See the per-(bench, metric) table above; treat any")
        print("  positive signal as suggestive pending more completed variants.")
print("=" * 72)
